In [9]:
import numpy as np

def rk4(x0, f, t0, tf, n):
    """ Calcula la evolución del sistema o ecuación diferencial empleando como método de aproximación
        el método de Runge Kutta orden 4.
        :parametro x0: Estado inicial, condiciones iniciales
        :parametro f: Función de carga f(t,x)
        :parametro t0: Tiempo inicial
        :parametro tf: Tiempo final
        :parametro n: Cantidad de puntos
    """
    h = (tf - t0) / n
    t = [t0]
    x = [x0]
    for i in range(1, n+1):
        t.append(t[i - 1] + h)
        k1 = f(t[i-1], x[i-1])
        k2 = f(t[i-1] + h * 0.5, x[i - 1] + 0.5 * h * k1)
        k3 = f(t[i-1] + h * 0.5, x[i - 1] + 0.5 * h * k2)
        k4 = f(t[i-1] + h, x[i - 1] + h * k3)
        x.append(x[i - 1] + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6)
    return np.array(t), np.array(x)

In [11]:
f = lambda t, x: x**2
x0 = 1
t0 = 0
tf = 0.1
n = 2
t, x = rk4(x0, f, t0, tf, n)
print(t)
print(x)

[0.   0.05 0.1 ]
[1.         1.05263156 1.11111107]


In [13]:
def steepest_descent(f, grad, xo, n):
    """ Búsqueda del mínimo de la función de campo escalar f,
        dado su gradiente y un punto inicial para partir. Además,
        se define una cantidad máxima de iteraciones.
        @param f: Función a optimizar
        @param grad: Gradiente de la función a optimizar
        @param xo: Punto inicial
        @param n: Cantidad de pasos máximos
    """
    
    # Condiciones iniciales y parámetros
    x = xo
    
    for i in range(n):
        # Calculo la dirección de descenso
        d = -grad(x)
        
        # Busqueda lineal por aproximación cuadrática de la longitud del paso
        g = lambda alfa: f(x + alfa * d)
        alfa_min, h_min = quadratic_aproximation(g, 0, 1)
        
        # Calculo la siguiente posición
        x = x + alfa_min * d
        
        print(f'Iteración {i+1}: x({i+1}) = {x}')
    
    # Retorna posición final
    return x

def quadratic_aproximation(f, xo, ho):
    """ Búsqueda del mínimo de una función mediante búsqueda lineal utilizando
        una interpolación cuadrática, donde se itera hasta encontrar una sonrisa,
        y luego aproximamos el salto.
        @param f: La función cuyo minimo buscamos
        @param xo: El punto inicial del intervalo
        @param ho: El tamaña del salto
    """
    # Definimos los parámetros y condiciones iniciales
    po = (xo, f(xo))
    p1 = (po[0] + ho, f(po[0] + ho))
    p2 = (p1[0] + ho, f(p1[0] + ho))
    
    # Iteramos en la búsqueda del mínimo, en realidad,
    # buscamos formar una sonrisa!
    found = False
    while not found:
        print(f'Iterando ({po[0]}, {po[1]}) - ({p1[0]}, {p1[1]}) - ({p2[0]}, {p2[1]})')
        if po[1] <= p1[1] <= p2[1]:
            ho = ho / 2
            p2 = p1
            p1 = (po[0] + ho, f(po[0] + ho))
        elif p2[1] <= p1[1] <= po[1]:
            ho = ho * 2
            p1 = p2
            p2 = (p1[0] + ho, f(p1[0] + ho))
        else:
            found = True
    
    # Cuando lo encontramos, establecemos una aproximación
    hmin = ho * (4 * p1[1] - 3 * po[1] - p2[1]) / (4 * p1[1] - 2 * po[1] - 2 * p2[1])
    return po[0] + hmin, hmin

In [15]:
f = lambda x: x[0]**2 + (x[1]-2)**2 + (x[2] + 3)**2
grad = lambda x: np.array([2*x[0],2*(x[1]-2),2*(x[2]+3)])
xo = np.array([0, 0, 0])
n = 1
steepest_descent(f, grad, xo, n)

Iterando (0, 13) - (1, 13) - (2, 117)
Iterando (0, 13) - (0.5, 0.0) - (1, 13)
Iteración 1: x(1) = [ 0.  2. -3.]


array([ 0.,  2., -3.])

In [22]:
from numpy import *
import numpy.polynomial.polynomial as poly

def lagrange_interpolation(a, b, n, f, cheby):
    """
        Encuentra el polinomio interpolador de 'f' en [a, b] usando polinomios de lagrange de orden n
        @param cheby: True si se usan nodos de chebyshev, de lo contrario equiespaciados
        @return (polinomio, (nodos_x, f(nodos_x)))
            Los coeficientes del polinomio son: [an,..., a1, a0] que el orden en que np.polyval() recibe los coeficientes 
            
        ¡Esta función fue realizada por Rafael Nicolás Trozzo, yo simplemente la copie y pegué.!
    """
    if cheby:
        zj = [np.cos((2*j+1)/(2*(n+1))*np.pi) for j in range(n+1)]
        x_nodes = [(b+a)/2 + (b-a)/2*z for z in zj] 
    else:
        x_nodes = np.linspace(a, b, n + 1)
    y_nodes = [f(x) for x in x_nodes]
    # hay n+1 polinomios con n+1 coeficientes
    lag_pols = np.zeros((n + 1, n + 1)) 
    for k in range(len(x_nodes)):
        # Formamos el polinomio de lagrange k a partir de sus raices
        lag_roots = np.delete(x_nodes, k)
        lag_pols[k] = np.flip(poly.polyfromroots(lag_roots)) / np.prod(x_nodes[k] - lag_roots) * y_nodes[k]
        print(lag_pols[k])  # comentar si no se quieren ver los polinomios de Lagrange
    interp_poly = np.sum(lag_pols, axis=0)
    
    return (interp_poly, (x_nodes, y_nodes))

In [28]:
a = -1
b = 1
n = 2
f = lambda x: np.exp(-abs(x))

lagrange_interpolation(a, b, n, f, True)

[ 2.80413351e-01  2.42845085e-01 -1.48699728e-17]
[-1.33333333 -0.          1.        ]
[ 2.80413351e-01 -2.42845085e-01  1.48699728e-17]


(array([-0.77250663,  0.        ,  1.        ]),
 ([0.8660254037844387, 6.123233995736766e-17, -0.8660254037844387],
  [0.42062002605411475, 0.9999999999999999, 0.42062002605411475]))

In [25]:
def newton_raphson(f: callable, f_: callable, x0: float, iter_count: int):
    """ Resolver ecuaciones no lineales por metodo de Newton Raphson
        @param f: Función del problema
        @param f_: Derivada de la función
        @param x0: Condicion inicial
        @param iter_count: Cantidad de pasos
    """
    x = [x0]
    for i in range(1, 1 + iter_count):
        x_i = x[i - 1] - f(x[i - 1])/f_(x[i - 1])
        x.append(x_i)
    return x

In [27]:
f = lambda x: np.log(x) - x / 4
f_ = lambda x: (1 / x) - (1 / 4)
x0 = 8.5
iter_count = 4
newton_raphson(f, f_, x0, iter_count)

[8.5, 8.613833235305156, 8.613169478615234, 8.6131694564414, 8.6131694564414]